# Lab 7: Local MCP — Connect to a Locally-Run MCP Server

**Difficulty: Intermediate | ~40 min | Requires Labs 1–2**

Your agent gains capabilities from **MCP** (Model Context Protocol) servers. This lab covers the
**local variant**: a small MCP server running on your own machine, connected over stdio — the easier,
lower-stakes half of MCP. You'll run a personal notes server, watch its tools become ordinary
LangChain tools, and drive them with the free Nemotron model.

## Step 1 — Install the required modules

One pinned command installs the LangChain stack from the previous labs plus the two new pieces:
`mcp` (the protocol SDK, which provides `FastMCP` for the server) and `langchain-mcp-adapters`
(the MCP client that converts a server's tools into LangChain tools).

In [1]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install -qU langchain==1.3.15 langchain-core==1.5.4 langchain-openai==1.4.3 langgraph==1.2.11 langchain-mcp-adapters==0.3.2 mcp==1.29.0 python-dotenv==1.2.2

## Step 2 — Load the key

Same as every lab since Lab 4: the OpenRouter key lives in a `.env` file in this folder. If you
haven't already, copy `.env.example` to `.env` and paste your real key. `load_dotenv()` reads it
into the environment.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Step 3 — Create the model

The same OpenRouter wrapper as Labs 1–6: `model=` names the free Nemotron model on OpenRouter,
`base_url=` redirects the OpenAI-compatible client to OpenRouter, `api_key=` pulls the key from
the environment, and `temperature=0` keeps answers deterministic.

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

## Step 4 — Meet the local MCP server

An MCP *server* is a program that exposes tools over the MCP protocol. This lab ships one in
`mcp_notes_server.py` (open it in your editor): a personal notes store backed by a JSON file. The
core is tiny — the SDK does the protocol work:

```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("notes-server")

@mcp.tool()
def add_note(title: str, content: str) -> str:
    """Add a note and return its id. The id is a sequential number."""
    ...
```

`@mcp.tool()` registers the function as a tool: the docstring becomes the description (the model
reads it to decide when to call), and the signature becomes the argument schema. The server knows
nothing about LangChain or models — it just answers MCP requests over stdio (standard input/output),
which is why this is the **local** variant: it runs as a subprocess on this machine. The file also
quiets the SDK's own request logs so the notebook output stays readable.

## Step 5 — Connect to the server and list its tools

`MultiServerMCPClient` is the bridge. Its constructor takes a dict of connections: one entry per
server, naming the server, the transport (`stdio`), the executable that runs it (`sys.executable` —
the Python running this notebook), and the server script as an argument. `await client.get_tools()`
spawns the server, asks it over the pipe which tools it exposes (an MCP `ListTools` request), and
returns LangChain `BaseTool`s. This step makes **no model calls** — the server is local and free.

In [4]:
import sys
from pathlib import Path

from langchain_mcp_adapters.client import MultiServerMCPClient

Path("notes.json").unlink(missing_ok=True)  # start from a clean notes store

client = MultiServerMCPClient(
    {
        "notes": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(Path("mcp_notes_server.py").resolve())],
        }
    }
)

tools = await client.get_tools()
for tool in tools:
    print(f"- {tool.name}: {tool.description.splitlines()[0]}")

- add_note: Add a note and return its id. The id is a sequential number.
- list_notes: List all notes as one line per note (id + title).
- get_note: Return one note by its id, with title and content.
- delete_note: Delete a note by its id. Returns a confirmation.


## Step 6 — See the schema the model sees

Each returned tool carries `args`: the JSON schema the model must fill in to call it. Print
`add_note`'s — it's the exact shape MCP delivered over `ListTools`, converted by the adapter.

In [5]:
print(tools[0].args)

{'title': {'title': 'Title', 'type': 'string'}, 'content': {'title': 'Content', 'type': 'string'}}


## Step 7 — Call MCP tools directly, no model involved

MCP tools are ordinary functions over a protocol — they don't need an LLM. Pull two tools out of
the list and call them with `.ainvoke()` (async invoke). Every call spawns the server, runs the
tool against `notes.json`, and returns a result. Watch `notes.json` appear in this folder. Zero
API calls.

In [6]:
add = [t for t in tools if t.name == "add_note"][0]
list_notes = [t for t in tools if t.name == "list_notes"][0]

print(await add.ainvoke({"title": "groceries", "content": "milk, eggs, bread"}))
print(await list_notes.ainvoke({}))

[{'type': 'text', 'text': 'Added note 1: groceries', 'id': 'lc_bca3fd9c-ef7c-4473-9893-9e3120640c33'}]
[{'type': 'text', 'text': '[1] groceries', 'id': 'lc_180287bf-8af9-436e-b840-b97db9e66874'}]


## Step 8 — Give the agent the MCP tools

`create_agent` is the same factory from Labs 1–6. Pass it the model and the MCP-derived tools: the
agent loop now treats the server's tools exactly like any LangChain tool. This query asks the agent
to add a note *and then* list all notes — one model call, two tool calls, and the tools run locally
for free. The printed messages show the whole loop: `human` → `ai` (the tool calls it decided on)
→ `tool` (the results) → `ai` (the answer).

In [7]:
from langchain.agents import create_agent

agent = create_agent(model=model, tools=tools)

result = await agent.ainvoke(
    {"messages": [("human", "Add a note titled 'meeting' with content 'standup at 10am', then list all my notes.")]}
)
for message in result["messages"]:
    print(f"{message.type}: {str(message.content)[:90]}")

human: Add a note titled 'meeting' with content 'standup at 10am', then list all my notes.
ai: 
tool: [{'type': 'text', 'text': 'Added note 2: meeting', 'id': 'lc_dfb37ec3-e5ad-4659-9734-5fc53
ai: 
tool: [{'type': 'text', 'text': '[1] groceries\n[2] meeting', 'id': 'lc_6595d4d3-172c-4a6c-b921-
ai: 

I've added the note titled "meeting" with content "standup at 10am". Here are all your n


## Step 9 — A fresh turn reads the persisted store

The notes live in `notes.json`, not in the model's memory. Ask the agent a follow-up in a fresh
invocation (same agent, new turn): it reaches for `list_notes`, the server reads the file, and the
answer reflects everything added in Steps 7–8.

In [8]:
result = await agent.ainvoke({"messages": [("human", "What notes do I have?")]})
for message in result["messages"]:
    print(f"{message.type}: {str(message.content)[:90]}")

human: What notes do I have?
ai: 
tool: [{'type': 'text', 'text': '[1] groceries\n[2] meeting', 'id': 'lc_f213dc54-d2ae-48cc-a2b0-
ai: 
You currently have 2 notes:

1. groceries
2. meeting


## Step 10 — The loop when no tool fits

Not every question needs a tool. Ask a general-knowledge question and watch the loop skip the tool
node entirely — one `ai:` message, no `tool:` line. The agent isn't forced to use its tools.

In [9]:
result = await agent.ainvoke({"messages": [("human", "In one sentence, what is MCP?")]})
for message in result["messages"]:
    print(f"{message.type}: {str(message.content)[:90]}")

human: In one sentence, what is MCP?
ai: 

MCP (Model Context Protocol) is an open standard that enables AI models to securely conn


## Step 11 — Delete through language

Capabilities include removal. Ask the agent to delete the groceries note and tell you what remains;
the model picks `delete_note` (then `list_notes`) from the tool descriptions alone.

In [10]:
result = await agent.ainvoke(
    {"messages": [("human", "Delete the note titled 'groceries', then tell me what notes remain.")]}
)
for message in result["messages"]:
    print(f"{message.type}: {str(message.content)[:90]}")

human: Delete the note titled 'groceries', then tell me what notes remain.
ai: 
tool: [{'type': 'text', 'text': '[1] groceries\n[2] meeting', 'id': 'lc_60937f01-868c-43ef-a97c-
ai: 
tool: [{'type': 'text', 'text': 'Deleted note 1.', 'id': 'lc_cc63ece7-ed28-4cf1-b72d-37e614bcfcb
ai: 
tool: [{'type': 'text', 'text': '[2] meeting', 'id': 'lc_e99a27e2-6973-4b0e-8acb-0613ebab79ae'}]
ai: 

I've deleted the note titled 'groceries'. The remaining note is:

[2] meeting


## Step 12 — Errors and server metadata

Two quick locals. A lookup for a note that doesn't exist returns a clean message from the tool (no
crash — the server says so in plain text). `get_server_info()` asks the server to identify itself
over the protocol.

In [11]:
get = [t for t in tools if t.name == "get_note"][0]
print(await get.ainvoke({"note_id": "99"}))

info = await client.get_server_info()
print(info["notes"].serverInfo.name, info["notes"].serverInfo.version)

[{'type': 'text', 'text': 'No note with id 99.', 'id': 'lc_f3769c9f-83e5-46f3-9839-2b1663e2f403'}]
notes-server 1.29.0


## Step 13 — Two servers, prefixed names

Connect the *same* server twice under different names. Without care, both would expose `add_note`
— a name collision. `tool_name_prefix=True` prefixes every tool with its server name, so the agent
can address `notes_add_note` and `work_add_note` unambiguously. This is how production clients
attach multiple MCP servers without their tools fighting.

In [12]:
client2 = MultiServerMCPClient(
    {
        "notes": {"transport": "stdio", "command": sys.executable, "args": [str(Path("mcp_notes_server.py").resolve())]},
        "work": {"transport": "stdio", "command": sys.executable, "args": [str(Path("mcp_notes_server.py").resolve())]},
    },
    tool_name_prefix=True,
)

all_tools = await client2.get_tools()
for tool in all_tools:
    print(f"- {tool.name}")

- notes_add_note
- notes_list_notes
- notes_get_note
- notes_delete_note
- work_add_note
- work_list_notes
- work_get_note
- work_delete_note


## Wrap-up

Every time the agent (or you) calls one of these tools, a stdio subprocess runs `mcp_notes_server.py`,
the client and server exchange MCP JSON-RPC messages (`ListTools`, `CallTool`), and the tool's Python
body runs. The protocol is what makes this powerful: **any** server that speaks MCP — local or remote —
drops into an agent the same way. The next lab covers the other half: a remotely-hosted MCP server.

Now try the Optional Exercise in the lab markdown: add a `search_notes` tool to the server and ask
the agent to find the standup note — no agent code changes needed.